# 04. 0단계 남은 점검

`plan.md` [0단계] 의 남은 점검 5개. 01 출력(`multiclass/`)과 `patterns.parquet` 만 읽는다.

| 절 | 점검 | 어디에 쓰나 |
|---|---|---|
| 1 | 엣지 1~2개짜리 작은 시도 | 구조로 구분 불가능한 엣지의 정확한 양 → 하한 표 |
| 2 | 분할 끝까지 안 닫힌 CYCLE | 닫히기 전 CYCLE = 경로 = RANDOM 모양 → 하한 표의 CYCLE→RANDOM 칸 |
| 3 | new 시도가 학습 계좌를 공유하나 | 테스트 new 묶음이 얼마나 깨끗한가 |
| 4 | UNMATCHED 가 속한 연결 요소 크기 | UNMATCHED 가 혼자인지, 작게 묶이는지, 거대 덩어리에 붙는지 |
| 5 | 요일·시간대별 클래스 분포 | "UNMATCHED 에 주간 주기" 추정 확인 → 요일·시간대 피처를 넣을지 |

## 0. 설정

In [ ]:
import sys
from pathlib import Path
import pandas as pd
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 220)

sys.path.append(str(Path.cwd()))
import mc_lib as M

DATASET = "HI-Large"
TRAIN = ("2022-08-08", "2022-10-02")
VAL   = ("2022-10-03", "2022-10-16")
TEST  = ("2022-10-17", "2022-10-30")
CUTOFF = TEST[1]

windows = M.make_windows(TRAIN, VAL, TEST)
OUT = M.results_dir(DATASET, "04_remaining_checks")
lau = M.load_laundering(DATASET)
att = M.load_attempts(DATASET)
daily = M.load_daily(DATASET)
print(f"세탁 엣지 {len(lau):,}  시도 {len(att):,}")

## 1. 엣지 1~2개짜리 작은 시도

- `attempts` : 패턴별 시도 중 전체 엣지 수가 1개 / 2개 / 2개 이하인 비율(%)
- `edges` : 분할 × 패턴 엣지 중
  - `full_le_%` = 시도 전체가 엣지 2개 이하 (끝까지 가도 구조가 없음)
  - `visible_le_%` = 분할 끝까지 보이는 시도 엣지가 2개 이하 (그 시점 그래프에서 구조가 없음)
  - `visible_1_%` = 분할 끝까지 보이는 엣지가 자기 자신뿐

In [ ]:
sm = M.small_attempts(lau, att, windows)
sm["attempts"]

In [ ]:
sm["edges"]

## 2. CYCLE 닫힘

- `shape` : 가정 확인. CYCLE 의 `edges_eq_accounts_%`(단순 사이클), RANDOM 의 `edges_eq_accounts-1_%`(경로)가 100 에 가까우면 "시도 엣지가 전부 보이면 닫힘" 판단이 맞다
- `closed` : 분할별 CYCLE 엣지 중 분할 끝까지 사이클이 닫힌 비율. `open_%` = 그래프에서 RANDOM(경로)과 같은 모양인 CYCLE 엣지

In [ ]:
cy = M.cycle_closure(lau, att, windows)
cy["shape"]

In [ ]:
cy["closed"]

## 3. new 시도의 학습 계좌 공유

- `attempts` : val·test 에서 새로 시작한 시도 중 학습 구간 세탁 거래의 계좌를 하나라도 포함하는 비율 (`share_train_acct_%`), 학습 **패턴** 거래 계좌를 포함하는 비율 (`share_train_pattern_acct_%`)
- `edges` : new 패턴 엣지 / UNMATCHED 엣지 중 양끝 하나가 학습 구간 세탁 계좌인 비율
- 높으면 new 묶음도 계좌를 통해 학습과 이어져 있다 → 계좌 단위 기억으로 맞힐 여지

In [ ]:
ns = M.new_attempt_sharing(lau, att, windows)
ns["attempts"]

In [ ]:
ns["edges"]

## 4. UNMATCHED 가 속한 연결 요소 크기 (테스트 끝 기준, 방향 무시)

- 행 = 요소 크기(엣지 수), 열 = 요소 종류 (UNMATCHED 만 / 시도 1개 + UNMATCHED / 여러 시도), 값 = UNMATCHED 엣지 수 · 비율

In [ ]:
cc = M.components(lau, CUTOFF)
us = M.unmatched_component_sizes(cc["edges"])
us["counts"]

In [ ]:
us["share"]

## 5. 요일 · 시간대별 클래스 분포 (학습~테스트 구간)

- `weekday` : 클래스 × 요일 비율(%), 행 합 100. 맨 아래 `NORMAL(all tx)` = 같은 기간 정상 거래 전체
- `weekday_ratio` : 균등(14.29%) 대비 배수. 1 에서 멀수록 요일 신호가 있다
- `hour` : 클래스 × 시간대 비율(%)

In [ ]:
wh = M.weekday_hour(lau, windows, daily)
wh["weekday"]

In [ ]:
wh["weekday_ratio"]

In [ ]:
wh["hour"]

In [ ]:
fig = M.plot_weekday_hour(wh, path=OUT / "weekday_hour.png")

## 6. 저장

In [ ]:
for name, df in {"small_attempts": sm["attempts"], "small_edges": sm["edges"],
                 "cycle_shape": cy["shape"], "cycle_closed": cy["closed"],
                 "new_sharing_attempts": ns["attempts"], "new_sharing_edges": ns["edges"],
                 "unmatched_comp_counts": us["counts"], "unmatched_comp_share": us["share"],
                 "weekday": wh["weekday"], "weekday_ratio": wh["weekday_ratio"], "hour": wh["hour"]}.items():
    df.to_csv(OUT / f"{name}.csv", encoding="utf-8-sig")
sorted(p.name for p in OUT.iterdir())